# Task 3, Step 7: Workload and Cost Dependence, Standalone Accuracy, Deployment Break-Even

* Speculative outcomes by workload (mathematics / familiar extraction / shifted-template extraction) and by actual
  output-length bin; paired bootstrap 95% intervals over prompts for throughput and acceptance differences (training
  seeds reported separately).
* Expected acceptance on the common prefix banks vs empirical acceptance on each draft's speculative trajectories.
* Standalone draft accuracy with answer-appropriate caps (512 math / 32 extraction), temperature 1, two repetitions:
  math answer extraction and extraction-format compliance. This ranks drafts *as models*; speculative output follows
  the target and is never used for this ranking.
* Deployment break-even: upfront distillation + merging cost divided by the measured per-request saving (or never,
  if the draft slows generation down).
* Representative matched-prompt continuations.

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task3_speculative")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2" or _ver("peft") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5", "peft>=0.13"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task3", need_public=True, need_prefix_banks=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from shared.src.io import read_jsonl, load_json, save_json
from shared.src.plotting import save_figure
from shared.src.stats import paired_bootstrap_diff, paired_bootstrap_ratio
from task3_speculative.src.data import test_prompts

OUT = ctx.results / "spec"
b = pd.DataFrame(read_jsonl(OUT / "benchmark_eos64.jsonl"))
DRAFTS = ["original", "off_s6304", "off_s6305", "on_s6304", "on_s6305"]

## By workload and output-length bin

In [ ]:
cols = ["acc_per_examined", "acc_per_proposed", "mean_accepted_per_verification", "target_passes_per_token", "tokens_per_s",
        "base_tokens_per_s", "speedup_tps", "n_out", "eos"]
wl = b.groupby(["gamma", "workload", "draft"])[cols].mean().round(3)
wl.to_csv(OUT / "by_workload.csv")
wl

In [ ]:
b["len_bin"] = pd.cut(b.n_out, [0, 8, 16, 32, 64], labels=["1-8", "9-16", "17-32", "33-64"])
lb = b.groupby(["gamma", "len_bin", "draft"], observed=True)[["acc_per_examined", "speedup_tps", "target_passes_per_token"]].mean().round(3)
lb.to_csv(OUT / "by_output_length.csv")
lb.unstack("draft")

## Paired bootstrap over prompts (trained draft vs original, and on- vs off-policy, per seed)

In [ ]:
def per_prompt(d, g, col):
    x = b[(b.draft == d) & (b.gamma == g)].groupby("pid")[col].mean()
    return x

rows = []
for g in (2, 4):
    for a, c in [(d, "original") for d in DRAFTS[1:]] + [("on_s6304", "off_s6304"), ("on_s6305", "off_s6305")]:
        for col in ("acc_per_examined", "tokens_per_s", "speedup_tps", "target_passes_per_token"):
            xa, xc = per_prompt(a, g, col), per_prompt(c, g, col)
            idx = xa.index.intersection(xc.index)
            m, lo, hi = paired_bootstrap_diff(xa[idx].values, xc[idx].values, n_boot=2000)
            rows.append(dict(gamma=g, A=a, B=c, metric=col, diff=m, lo=lo, hi=hi, n_prompts=len(idx)))
boot = pd.DataFrame(rows)
boot.to_csv(OUT / "paired_bootstrap.csv", index=False)
boot.pivot_table(index=["gamma", "A", "B"], columns="metric", values="diff").round(4)

## Expected (prefix-bank) vs empirical (trajectory) acceptance

In [ ]:
met = pd.DataFrame(read_jsonl(ctx.results / "alignment" / "prefix_metrics.jsonl.gz"))
exp_acc = met[met.bank == "teacher"].groupby("draft").exp_accept.mean()
emp = b.groupby(["draft", "gamma"]).apply(lambda x: x.accepted.sum() / x.examined.sum()).unstack("gamma")
cmp = pd.DataFrame({"expected_1-TV_teacher_bank": exp_acc, "empirical_gamma2": emp[2], "empirical_gamma4": emp[4],
                    "fkl_teacher_bank": met[met.bank == "teacher"].groupby("draft").fkl.mean(),
                    "tv_teacher_bank": met[met.bank == "teacher"].groupby("draft").tv.mean()}).loc[DRAFTS]
cmp.to_csv(OUT / "expected_vs_empirical_acceptance.csv")
ax = cmp.plot.scatter("expected_1-TV_teacher_bank", "empirical_gamma4", figsize=(5, 4))
for d, r in cmp.iterrows():
    ax.annotate(d, (r["expected_1-TV_teacher_bank"], r["empirical_gamma4"]), fontsize=8)
save_figure(ctx.figures, "expected_vs_empirical_acceptance")
cmp.round(4)

## Standalone draft accuracy (the draft as a model)

In [ ]:
from task3_speculative.src.distill import load_models, load_merged
from task3_speculative.src.spec_eval import standalone_accuracy
acc_path = OUT / "standalone_accuracy.jsonl"
M = load_models(load_target=False)
tok = M["tok"]
models = {"original": M["draft"], **{d: load_merged(ctx.checkpoints / "merged" / d) for d in DRAFTS[1:]}}
for name, m in models.items():
    standalone_accuracy(m, tok, acc_path, name)
sa = pd.DataFrame(read_jsonl(acc_path))
sa_tab = sa.groupby(["model", "workload"]).agg(acc=("correct", "mean"), contains=("contains", "mean"), extractable=("extractable", "mean"),
                                                truncated=("truncated", "mean"), mean_len=("n_out", "mean")).unstack("workload").round(3)
sa_tab.to_csv(OUT / "standalone_accuracy_table.csv")
sa_tab

## Deployment break-even including distillation and merging cost

In [ ]:
costs = pd.read_csv(ctx.results / "distill" / "training_costs.csv")
pool = load_json(ctx.results / "distill" / "teacher_pool_stats.json")
rows = []
for d in DRAFTS[1:]:
    method, seed = d.split("_s")
    c = costs[costs.run == f"{method}_policy_s{seed}"].iloc[0]
    merge = load_json(ctx.results / "distill" / f"merge_{d}.json")["merge_time_s"]
    upfront = c.wall_s + merge + (pool["teacher_generation_s"] if method == "off" else 0.0)
    for g in (2, 4):
        x = b[(b.draft == d) & (b.gamma == g)]
        o = b[(b.draft == "original") & (b.gamma == g)]
        save_vs_target = (x.base_total_ms - x.total_ms).mean() / 1e3            # seconds saved per request vs target-only
        save_vs_orig = (o.total_ms.mean() - x.total_ms.mean()) / 1e3            # vs the undistilled draft
        rows.append(dict(draft=d, gamma=g, upfront_s=upfront, saving_vs_target_s=save_vs_target, saving_vs_original_draft_s=save_vs_orig,
                         requests_to_amortise_vs_original=(upfront / save_vs_orig) if save_vs_orig > 0 else np.inf))
be = pd.DataFrame(rows)
be.to_csv(OUT / "deployment_break_even.csv", index=False)
be.round(3)

## Representative matched-prompt continuations (fixed rule: first math, familiar and shifted prompt, gamma = 4, rep 0)

In [ ]:
tp = {p["pid"]: p for p in test_prompts()}
pick = [next(p["pid"] for p in test_prompts() if p["workload"] == w) for w in ("math", "short_familiar", "short_shifted")]
for pid in pick:
    print(f"\n=== {pid} [{tp[pid]['workload']}] gold={tp[pid]['answer']}\n{tp[pid]['prompt'][:300]}")
    for d in DRAFTS:
        r = b[(b.pid == pid) & (b.draft == d) & (b.gamma == 4) & (b.rep == 0)]
        if len(r):
            r = r.iloc[0]
            print(f"  [{d:>10}] acc={r.acc_per_examined:.2f} passes/tok={r.target_passes_per_token:.2f} speedup={r.speedup_tps:.2f} :: "
                  + tok.decode(r.out_tokens, skip_special_tokens=True)[:160].replace("\n", " "))